# Project 08: Fintech Credit Risk Analysis
### Behavioral Risk Scoring — Pillar 1

**Notebook:** 02_risk_scoring.ipynb
**Phase:** 2 — Borrower Default Risk (behavioral scoring component)
**Input:** data/processed/cs_training_cleaned.csv
**Author:** Taofeek Salami
**Date:** 2026-07-14

**Methodological note:** Because the dataset lacks event timestamps,
delinquency buckets are used as behavioral proxies rather than true
temporal recency. This creates a practical behavioral risk score rather
than a traditional RFM model.

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/cs_training_cleaned.csv')
print(df.shape)

(149999, 14)


## Behavioral Risk Score — Component 1: Frequency

Frequency is the total count of late-payment incidents across all three
delinquency fields (30-59, 60-89, and 90+ days past due). Given the
0.98–0.99 correlation observed between these fields in Phase 1 EDA, this
sum captures a single underlying "repeat delinquency" behavior rather
than three independent signals.

In [2]:
df['frequency_score'] = (
    df['NumberOfTime30_59DaysPastDueNotWorse'] +
    df['NumberOfTime60_89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

print(df['frequency_score'].describe())

count    149999.000000
mean          0.927393
std          12.466246
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         294.000000
Name: frequency_score, dtype: float64


## Behavioral Risk Score — Component 2: Recency (Proxy)

No event timestamps exist in this dataset, so recency is proxied using
the *severity tier* of delinquency bucket reached — 90+ days is treated
as a stronger/more recent-acting signal than 30-59 days. Borrowers are
scored 0-3 based on the most severe bucket they've ever recorded a
delinquency in.

In [3]:
def recency_proxy(row):
    if row['NumberOfTimes90DaysLate'] > 0:
        return 3
    elif row['NumberOfTime60_89DaysPastDueNotWorse'] > 0:
        return 2
    elif row['NumberOfTime30_59DaysPastDueNotWorse'] > 0:
        return 1
    else:
        return 0

df['recency_proxy_score'] = df.apply(recency_proxy, axis=1)
print(df['recency_proxy_score'].value_counts().sort_index())

recency_proxy_score
0    119637
1     17213
2      4811
3      8338
Name: count, dtype: int64


## Behavioral Risk Score — Component 3: Severity (Debt Ratio)

Debt ratio serves as the financial severity dimension. Since debt ratio
contains extreme outliers (values in the thousands, as seen in Phase 1
EDA), it is binned into quartiles rather than used as a raw continuous
score, keeping it on a comparable 0-3 scale with the other two components.

In [4]:
df['severity_score'] = pd.qcut(df['DebtRatio'], q=4, labels=[0, 1, 2, 3]).astype(int)
print(df['severity_score'].value_counts().sort_index())

severity_score
0    37500
1    37500
2    37499
3    37500
Name: count, dtype: int64


In [5]:
# Check for suspiciously high delinquency counts (likely data entry / placeholder codes)
print(df['NumberOfTime30_59DaysPastDueNotWorse'].value_counts().sort_index(ascending=False).head(10))
print(df['NumberOfTime60_89DaysPastDueNotWorse'].value_counts().sort_index(ascending=False).head(10))
print(df['NumberOfTimes90DaysLate'].value_counts().sort_index(ascending=False).head(10))

NumberOfTime30_59DaysPastDueNotWorse
98    264
96      5
13      1
12      2
11      1
10      4
9      12
8      25
7      54
6     140
Name: count, dtype: int64
NumberOfTime60_89DaysPastDueNotWorse
98    264
96      5
11      1
9       1
8       2
7       9
6      16
5      34
4     105
3     318
Name: count, dtype: int64
NumberOfTimes90DaysLate
98    264
96      5
17      1
15      2
14      2
13      4
12      2
11      5
10      8
9      19
Name: count, dtype: int64


In [6]:
# Do the 96/98 placeholder records show a different default rate than the rest?
placeholder_mask = (
    (df['NumberOfTime30_59DaysPastDueNotWorse'].isin([96, 98])) |
    (df['NumberOfTime60_89DaysPastDueNotWorse'].isin([96, 98])) |
    (df['NumberOfTimes90DaysLate'].isin([96, 98]))
)
print(placeholder_mask.sum())
print(df[placeholder_mask]['SeriousDlqin2yrs'].mean())
print(df[~placeholder_mask]['SeriousDlqin2yrs'].mean())

269
0.5464684014869888
0.06597876177118814


### Decision 5: Delinquency placeholder codes (96/98) — 269 records, 0.18%

Values of 96 or 98 appear identically across all three delinquency-count
fields for 269 borrowers — not organic counts (98 late payments would mean
being late almost every month for 8+ years), but a known placeholder/error
code in this dataset. These records are flagged via
`delinquency_data_error_flag` and excluded from the numeric `frequency_score`
calculation to avoid distorting its scale.

This is not treated as noise: borrowers with this flag show a **54.65%**
default rate vs. **6.60%** for the rest of the population — the strongest
single risk signal found in this dataset, stronger than the utilization
extreme flag (37.25%) or delinquency progression (18.13%). The flag itself
is carried forward as a standalone high-risk indicator in the composite
risk tier logic.

In [7]:
# Flag placeholder/error delinquency codes (96, 98) — confirmed as a strong
# risk signal (54.65% vs 6.60% default rate), not simple data noise.
df['delinquency_data_error_flag'] = (
    (df['NumberOfTime30_59DaysPastDueNotWorse'].isin([96, 98])) |
    (df['NumberOfTime60_89DaysPastDueNotWorse'].isin([96, 98])) |
    (df['NumberOfTimes90DaysLate'].isin([96, 98]))
)

# Recalculate frequency_score, excluding flagged records from the raw sum
# (set to the dataset's 95th percentile among non-flagged records instead,
# so flagged borrowers still score high without the placeholder value
# itself distorting the scale)
freq_cap = df[~df['delinquency_data_error_flag']]['frequency_score'].quantile(0.95)

df['frequency_score'] = df.apply(
    lambda row: freq_cap if row['delinquency_data_error_flag'] else row['frequency_score'],
    axis=1
)

print(df['delinquency_data_error_flag'].sum())  # should be 269
print(freq_cap)
print(df['frequency_score'].describe())

269
2.0
count    149999.000000
mean          0.403936
std           1.103238
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          19.000000
Name: frequency_score, dtype: float64


## Composite Behavioral Risk Score & Risk Tiers

The three components — frequency, recency proxy, and severity — are
combined into a single composite score. `frequency_score` is first binned
into quartiles (0-3) to match the scale of the other two components, so
no single dimension dominates due to differing raw ranges.

Borrowers with `delinquency_data_error_flag = True` are automatically
assigned to the Critical tier regardless of their computed score, given
the 54.65% default rate confirmed for this group — this flag is a
stronger standalone signal than the composite score would capture on
its own.

In [9]:
# frequency_score is heavily zero-inflated (75% of borrowers = 0 incidents),
# so quartile binning collapses. Using manual bins based on incident count instead.
def bin_frequency(score):
    if score == 0:
        return 0
    elif score == 1:
        return 1
    elif score <= 3:
        return 2
    else:
        return 3

df['frequency_binned'] = df['frequency_score'].apply(bin_frequency)

print(df['frequency_binned'].value_counts().sort_index())

frequency_binned
0    119637
1     17242
2      9095
3      4025
Name: count, dtype: int64


In [10]:
df['composite_risk_score'] = (
    df['frequency_binned'] + df['recency_proxy_score'] + df['severity_score']
)

print(df['composite_risk_score'].describe())
print(df['composite_risk_score'].value_counts().sort_index())

count    149999.000000
mean          2.162374
std           1.834386
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max           9.000000
Name: composite_risk_score, dtype: float64
composite_risk_score
0    30676
1    29902
2    30906
3    35145
4     6972
5     7329
6     3954
7     2663
8     1761
9      691
Name: count, dtype: int64


In [11]:
# Assign tiers based on composite score, with data-error flag override
def assign_tier(row):
    if row['delinquency_data_error_flag']:
        return 'Critical'
    elif row['composite_risk_score'] <= 1:
        return 'Low'
    elif row['composite_risk_score'] <= 3:
        return 'Medium'
    elif row['composite_risk_score'] <= 6:
        return 'High'
    else:
        return 'Critical'

df['risk_tier'] = df.apply(assign_tier, axis=1)

# Validate: does actual default rate increase cleanly across tiers?
tier_check = df.groupby('risk_tier')['SeriousDlqin2yrs'].agg(['count', 'mean'])
print(tier_check)

           count      mean
risk_tier                 
Critical    5356  0.471060
High       18014  0.195626
Low        60578  0.022335
Medium     66051  0.039757


In [12]:
df.to_csv('../data/processed/cs_training_risk_scored.csv', index=False)
print(df.shape)

(149999, 21)
